# PreprocesamientoEn este notebook hago la exploración inicial del dataset y defino cómo construyo el texto enriquecido que voy a usar en todos los modelos. Decidí concatenar los metadatos del orador como tokens al inicio del statement, separados con [SEP], porque así tanto el modelo lineal como los transformer pueden aprovechar esa información sin tener que codificarla aparte con one-hot.

In [ ]:
import pandas as pdimport numpy as npTRAIN_PATH = '/kaggle/input/2025-26-false-political-claim-detection/train.csv'TEST_PATH  = '/kaggle/input/2025-26-false-political-claim-detection/test_nolabel.csv'train = pd.read_csv(TRAIN_PATH)test  = pd.read_csv(TEST_PATH)print(train.shape, test.shape)train.head(3)

El dataset tiene 8.950 ejemplos de entrenamiento y 3.836 de test. Las columnas que me interesan son statement (el texto), speaker, party_affiliation, speaker_job, state_info y subject. La variable objetivo (label) solo está en train: 1 si la afirmación es falsa, 0 si es verdadera.

In [ ]:
train['label'].value_counts(normalize=True).round(3)

Hay desbalance: aproximadamente 65 % de la clase 1 (falsas) frente a 35 % de la clase 0. Lo tendré en cuenta más adelante con class weights o ajustando el umbral de decisión, en lugar de re-muestrear (lo que distorsionaría las probabilidades).

In [ ]:
# Miro qué oradores son más informativostop_speakers = (train.groupby('speaker')['label']                .agg(['count','mean'])                .sort_values('count', ascending=False)                .head(15))top_speakers

Algunos oradores tienen tasas de falsedad muy marcadas: chain-email cerca del 97 %, donald-trump alrededor del 83 %. Esto me convence de que el speaker es un predictor muy fuerte por sí solo, así que lo incluyo de forma explícita en el texto que paso al modelo en lugar de descartarlo.

In [ ]:
train.groupby('party_affiliation')['label'].agg(['count','mean']).sort_values('count', ascending=False).head(8)

Republican aparece en el 69,5 % de afirmaciones falsas dentro de su grupo, democrat en el 56,2 %. Es otra señal fuerte que vale la pena incluir como token contextual.

In [ ]:
# Construyo el texto enriquecido que voy a usar en todos los modelosdef build_text(row):    parts = []    for col, prefix in [('speaker','Speaker'), ('party_affiliation','Party'),                        ('speaker_job','Job'), ('state_info','State'),                        ('subject','Topic')]:        v = row.get(col)        if pd.notna(v) and str(v).strip() != '':            parts.append(f'{prefix}: {v}')    head = ' | '.join(parts)    return f"{head} [SEP] Claim: {row['statement']}"train['text_enriched'] = train.apply(build_text, axis=1)test['text_enriched']  = test.apply(build_text, axis=1)print(train['text_enriched'].iloc[0])

Cada fila queda como 'Speaker: donald-trump | Party: republican | Job: President-Elect | State: New York | Topic: china [SEP] Claim: ...'. Elegí el token [SEP] porque es el separador estándar del tokenizer de BERT/DeBERTa y le indica al modelo un cambio de contexto. La cabecera la pongo SIEMPRE antes del statement: si la pongo después, el truncamiento a 160 tokens del transformer la cortaría en los statements largos.

In [ ]:
from sklearn.model_selection import StratifiedKFoldskf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)folds = np.zeros(len(train), dtype=int)for f, (_, vi) in enumerate(skf.split(train, train['label'])):    folds[vi] = fpd.Series(folds).value_counts().sort_index()

Decidí fijar 5 folds estratificados con semilla 42 desde el principio, porque luego quiero comparar las predicciones OOF de modelos distintos sin que un fold contamine al siguiente. Si cambiase la semilla a media competencia perdería esa comparabilidad. Replico exactamente esta misma asignación al principio de cada notebook posterior.

In [ ]:
train['n_words'] = train['statement'].str.split().str.len()print(train['n_words'].describe(percentiles=[0.5, 0.9, 0.95]).round(1))

La mediana de longitud son unas 17 palabras y el percentil 95 está en torno a 35. Con MAX_LEN=160 tokens del tokenizer cubro de sobra el statement más toda la cabecera de metadatos, sin desperdiciar memoria de GPU.